# MABe Phase 12 offline inference
This notebook performs no downloads or package installation. Attach the Phase 12 bundle as a Kaggle Dataset, select the competition data, disable internet, and run all cells. The bundle may be nested below the Kaggle Dataset root.

In [ ]:
import json
import os
import sys
from importlib.metadata import version
from pathlib import Path

input_root = Path(os.environ.get('MABE_PHASE12_INPUT_ROOT', '/kaggle/input'))
working_root = Path(os.environ.get('MABE_PHASE12_WORK_ROOT', '/kaggle/working'))
manifests = sorted(input_root.rglob('bundle_manifest.json'))
if len(manifests) != 1:
    raise RuntimeError(f'Expected exactly one Phase 12 bundle, found {manifests}')
bundle_root = manifests[0].parent
sys.path.insert(0, str(bundle_root / 'src'))
from mabe.phase12_release import verify_phase12_bundle
verify_phase12_bundle(bundle_root)
print(f'Verified offline bundle: {bundle_root}')
print(
    f'Runtime: Python {sys.version.split()[0]}, Polars {version("polars")}, '
    f'XGBoost {version("xgboost")}, PyTorch {version("torch")}'
)

In [ ]:
from mabe.phase12_config import load_phase12_config
from mabe.phase12_inference import (
    discover_dataset_root,
    recover_submission_after_failure,
    run_offline_inference,
    write_emergency_submission,
)

config = load_phase12_config(bundle_root / 'configs/inference.yaml')
print(f'Inference version: {config.inference_version}')
dataset_override = os.environ.get('MABE_PHASE12_DATASET_ROOT')
dataset_root = discover_dataset_root(
    Path(dataset_override) if dataset_override else None
)
print(f'Discovered competition data: {dataset_root}')
submission_path = working_root / 'submission.csv'
inference_work_root = working_root / 'mabe_phase12'
try:
    write_emergency_submission(dataset_root, submission_path)
    print(f'Prepared emergency submission: {submission_path}')
except Exception as error:
    print(f'Emergency preparation warning: {type(error).__name__}: {error}')
try:
    outputs = run_offline_inference(
        config,
        dataset_root=dataset_root,
        work_root=inference_work_root,
        submission_path=submission_path,
    )
    print(outputs)
except Exception as error:
    print(f'Full inference recovered from {type(error).__name__}: {error}')
    failure_path = inference_work_root / 'top_level_failure.txt'
    failure_path.parent.mkdir(parents=True, exist_ok=True)
    failure_path.write_text(
        f'{type(error).__name__}: {error}\n', encoding='utf-8'
    )
    recover_submission_after_failure(
        dataset_root, inference_work_root, submission_path
    )
    print(f'Recovered submission: {submission_path}')
    raise RuntimeError(
        'Full hidden inference did not complete; refusing to silently score '
        'the emergency submission.'
    ) from error

In [ ]:
import polars as pl

submission = pl.read_csv(submission_path)
test_video_count = pl.read_csv(dataset_root / 'test.csv').height
print(f'Submission rows: {submission.height}; test metadata videos: {test_video_count}')
manifest_report_path = inference_work_root / 'manifests/manifest_build_report.json'
inference_report_path = inference_work_root / 'inference_report.json'
if manifest_report_path.is_file():
    manifest_report = json.loads(manifest_report_path.read_text(encoding='utf-8'))
    print('Manifest report:', manifest_report)
if inference_report_path.is_file():
    inference_report = json.loads(inference_report_path.read_text(encoding='utf-8'))
    print('Inference routes:', inference_report['inference_route_counts'])
    print('Failed videos:', inference_report['failed_video_count'])
else:
    print('WARNING: full inference report is absent; submission was recovered.')
submission.head()